# CASMI 2026 - Stage D: does the new stack actually win?

Scores the full phase1 ranking pipeline over the 1,145 `val_scripts` molecules in four
configurations:

| config | FPNet | rank_train.npz | what it isolates |
|---|---|---|---|
| `base`       | old | old | your current leaderboard anchor |
| `model_only` | **new** | old | the coupling trap - **expect flat or negative** |
| `matrix_only`| old | **new** | did regenerating the matrix alone help? |
| `both`       | **new** | **new** | the shipping candidate |

**Decision rule.** Ship `both` only if C2 and C2_hard each gain more than 1 SE *and* C1_npx
does not regress by more than 1 SE. At n=400 and n=200, 1 SE on MRR is roughly 0.02 and 0.03.
Anything smaller is noise and chasing it costs a day for nothing.

If `model_only` lands flat-to-negative while `both` gains, that is not a bug - it is the
coupling hypothesis confirming itself, and it is the reason Stage C exists.

### Efficiency
Channels 1, 2 and 3 do not depend on the FPNet or on the ranker, so they are computed **once**
and cached. Only the Channel-4 logits (twice: old, new) and the final feature assembly and
GBDT scoring (four times) are repeated. That turns four full passes into roughly one.

### Attach
competition data, `casmi26-offline-rdkit-2026033`, `coconut_dataset`, `val_scripts`,
`casmi26-fp-models-v2` (old), `casmi26-ranker-features` (old), `casmi26-fpnet-v3` (new).
Here you *want* both model datasets attached - this notebook selects between them explicitly
by path rather than by glob.

## Cells 1-8 - verbatim from the inference notebook

In [ ]:
# ===================================================================================
#  CONFIG — Tuned Hyperparameters Annotated with Experimental Measurements
# ===================================================================================
class CFG:
    # --- Candidate Generation (Phase 1 Calibrated & Progressive) ------------------
    PPM_WIN      = 8.5    # Optimal base window for timsTOF (+1.4 ppm systematic offset calibrated).
                          # Tighter is better: +-10 ppm -> 0.521, +-8.5 ppm -> 0.524 Class-2 MRR
    PPM_CENTER   = 1.45   # D1b measured: +1.45 ppm offset (IQR [+0.56, +2.20]) of the measured
                          # neutral mass vs the label's exact mass. The query is DIVIDED by
                          # (1 + PPM_CENTER/1e6) so the window is CENTRED rather than widened.
    PPM_EXPAND   = (8.5, 20.0, 45.0, 80.0) # Multi-tier progressive window to guarantee >= 25 distinct candidates
    PPM_FALLBACK = 30.0   # Robust fallback if tight window yields 0 candidates (legacy compatibility)

    # --- Spectral Preprocessing ----------------------------------------------------
    INT_FLOOR    = 0.002  # Drop peaks below 0.2% base-peak intensity
    MAX_PEAKS    = 64     # D6 measured: 64 > 256 (0.9216 vs 0.9143 structure-indexed, floor 0.002)
    MZ_TOL       = 0.01   # Da tolerance for peak alignment
    INT_POWER    = 1.0    # Linear intensity with entropy weighting outperforms sqrt
    ENT_WEIGHT   = True   # Entropy weighting for low-entropy spectra (Li et al. 2021)

    # --- Mass-Shifted Analog Propagation (Channel 2) --------------------------------
    ANALOG_WIN   = 200.0  # +- Da mass-shift search window
    N_ANALOG     = 100    # Analogs kept per molecule (scaled from 80 -> 100)
    SIM_POWER    = 4.0    # sim^p power weighting. p=1 -> 0.498, p=3 -> 0.521, p=4 -> 0.525

    # --- Calibrated Ensemble Ranker (Variance Reduction) ---------------------------
    W1_PRIORS    = (0.30, 0.60)  # Hedging Class-1/Class-2 trade-off improves LB by ~0.003
    SEEDS        = (0, 1, 2, 3)  # Bagging 4 random seeds eliminates 0.006 seed noise floor
    W1           = 0.50          # Calibrated class weight anchor
    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03,
               min_samples_leaf=80, l2_regularization=1.0)

    USE_BIO_DB   = False  # ChEBI/LIPID MAPS toggle (disabled by default to prevent decoy dilution)
    TOPN         = 25     # Output exactly 25 SMILES per molecule

    # --- Channel 2 Polarity Enrichment ---------------------------------------------
    POLARITY_ANALOG = True   # Keep the richest representative per (scaffold, polarity) pair.
                             # This ENRICHES the analog rep set only - candidates are never
                             # masked out by query polarity, so coverage is always >= the
                             # one-rep-per-scaffold behaviour it replaces.

    # --- TIER-1 Restored Settings (all measured in diagnostics-results.ipynb) -------
    STRUCT_MASS  = True   # D4/D6: index the reference library on the LABEL's exact mass instead
                          # of neutral_mass(precursor_mz, adduct). riken reports m/z to 2 dp, so
                          # 30% of its 341k spectra fall outside a +-8.5 ppm window for a purely
                          # numeric reason. Worth +0.006 MRR; 2,401 NP structures become reachable.
    MAX_BONDS    = 60     # D3: at 34, 15.2% of candidates got ONLY the intact parent mass.
                          # Test masses reach 1,159 Da, so the real bail-out rate is far worse.
    DEDUP_KEYS   = True   # D5: 0.54% of submitted slots collapsed to a duplicate key.

    # --- Slot Filling (append-only; can never displace a tight-window hit) ----------
    EXPAND_BACKFILL = True   # Stage B: rank widened-window candidates and APPEND below stage A
    BACKFILL_PPM    = 120.0  # Stage C: final mass-nearest top-up instead of 'CCO' padding

    # --- Fast-Commit / Re-run Controls (fail-safe: FULL unless proven public) -------
    ALLOW_FAST         = True   # Master opt-in. False => always full run, detection bypassed.
    FAST_DEV_MOLECULES = 30     # Molecules deep-evaluated during a recognised fast commit
    PUBLIC_N_MOL       = 400    # Public test fingerprint: unique molecule_id count
    PUBLIC_N_SPEC      = 1213   # Public test fingerprint: spectrum (row) count


In [ ]:
import os, sys, glob, time, pickle, math, subprocess
from multiprocessing import Pool as MPool
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import matplotlib.pyplot as plt

T0 = time.time()

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f"File not found in /kaggle/input: {name}")
    return sorted(hits, key=len)[0]

COMP   = os.path.dirname(find_file('test.parquet'))
TRAIN  = os.path.join(COMP, 'train.parquet')
TEST   = os.path.join(COMP, 'test.parquet')
SAMPLE = os.path.join(COMP, 'sample_submission.csv')

print(f"[INFO] Competition directory: {COMP}")
print(f"       Files available: {os.listdir(COMP)}")

# Bootstrap offline RDKit wheel if available in attached datasets
rdkit_whls = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)
if rdkit_whls:
    print(f"[INFO] Installing offline RDKit wheel: {rdkit_whls[0]}")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', rdkit_whls[0]], check=False)

try:
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
    from rdkit.Chem.Descriptors import ExactMolWt
    RDLogger.DisableLog('rdApp.*')
    HAVE_RDKIT = True
    print("[SUCCESS] RDKit 2026.3.3 is active and verified.")
except Exception as e:
    HAVE_RDKIT = False
    print(f"[WARNING] RDKit unavailable ({e}). Running without in-silico fragmentation.")


In [ ]:
import numpy as np
from numba import njit, prange

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx: mx = it[i]
    if mx <= 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx
    c = 0
    for i in range(n):
        if it[i] >= thr: c += 1
    idx = np.empty(c, np.int64)
    j = 0
    for i in range(n):
        if it[i] >= thr:
            idx[j] = i; j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c): v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk): k2[i] = idx[o[i]]
        k2.sort()
        idx = k2
        c = topk
    om = np.empty(c, np.float32)
    oi = np.empty(c, np.float32)
    s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]
        v = it[idx[i]] ** power
        oi[i] = v
        s += v
    if s > 0:
        for i in range(c): oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0: S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S
            s2 = 0.0
            for i in range(c):
                oi[i] = oi[i] ** w
                s2 += oi[i]
            if s2 > 0:
                for i in range(c): oi[i] /= s2
    return om, oi

@njit(cache=True, fastmath=True)
def entropy_sim(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        d = qmz[i] - cmz[j]
        if d < -tol:
            buf[b] = qp[i]; i += 1; b += 1
        elif d > tol:
            buf[b] = cp[j]; j += 1; b += 1
        else:
            buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n: buf[b] = qp[i]; i += 1; b += 1
    while j < m: buf[b] = cp[j]; j += 1; b += 1
    for x in range(b): tot += buf[x]
    if tot <= 0: return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0: SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)

@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):
    """Evaluates max of direct and mass-shifted spectral entropy similarity."""
    a = entropy_sim(qmz, qp, cmz, cp, tol)
    if shift > -0.001 and shift < 0.001:
        return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)):
        sm[i] = cmz[i] + shift
    b = entropy_sim(qmz, qp, sm, cp, tol)
    return a if a > b else b

@njit(cache=True, fastmath=True, parallel=True)
def search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim(qmz, qp, cm, cp, tol)
    return out

@njit(cache=True, fastmath=True, parallel=True)
def search_shift(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])
    return out


In [ ]:
# ===================================================================================
#  High-Precision Adduct Physics & Reference Library Construction
# ===================================================================================
MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,
            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,
            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)
E = 0.00054857990
PROTON = MASS['H'] - E
H2O = 2 * MASS['H'] + MASS['O']
NH4 = MASS['N'] + 4 * MASS['H']
FORMATE = MASS['C'] + 2 * MASS['H'] + 2 * MASS['O']
ACETATE = 2 * MASS['C'] + 4 * MASS['H'] + 2 * MASS['O']

ADDUCTS = {
    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS['Na'] - E),
    "[M+K]+": (1, 1, MASS['K'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2*H2O),
    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),
    "[M+CH3OH+H]+": (1, 1, PROTON + MASS['C'] + 4 * MASS['H'] + MASS['O']),
    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS['C'] + 3 * MASS['H'] + MASS['N']),
    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),
    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS['Cl'] + E), "[M]-": (1, 1, E),
    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS['Na'] - 2 * PROTON),
    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS['Na'] - E), "[2M+NH4]+": (2, 1, NH4 - E),
    "[2M+K]+": (2, 1, MASS['K'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),
    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS['Na'] - 2 * PROTON),
    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),
}

# --- TIER-1: exact neutral mass from the labelled molecular formula -----------------
# Far faster than 275k RDKit parses and exactly what the label implies.
import re as _re
_MONO = {"C":12.0,"H":1.0078250319,"N":14.0030740052,"O":15.9949146221,"S":31.97207069,
         "P":30.97376151,"F":18.99840320,"Cl":34.96885271,"Br":78.9183376,"I":126.904468,
         "Si":27.9769265327,"Se":79.9165196,"B":11.0093055,"Na":22.98976928,"K":38.96370649,
         "Fe":55.9349375,"D":2.0141017778,"As":74.9215965,"Li":7.0160034,"Sn":119.9021947}
_TOK = _re.compile(r"([A-Z][a-z]?)(\d*)")
def formula_mass(f):
    if not isinstance(f, str): return np.nan
    s, pos = 0.0, 0
    for m in _TOK.finditer(f):
        if m.start() != pos: return np.nan
        pos = m.end()
        el = m.group(1)
        if el not in _MONO: return np.nan
        s += _MONO[el] * (int(m.group(2)) if m.group(2) else 1)
    return s if pos == len(f) else np.nan


def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan)
    ad = np.asarray(adduct, dtype=object)
    for a, (n, z, d) in ADDUCTS.items():
        m = (ad == a)
        if m.any():
            out[m] = (mz[m] * z - d) / n
    return out

def load_library(path):
    t0 = time.time()
    t = pq.read_table(path, columns=['inchikey14', 'normalized_smiles', 'adduct', 'ionization_mode',
                                     'precursor_mz', 'molecular_formula',
                                     'ms2_mzs', 'ms2_normalized_intensities'])
    mzc = t.column('ms2_mzs').combine_chunks()
    itc = t.column('ms2_normalized_intensities').combine_chunks()
    off = mzc.offsets.to_numpy().astype(np.int64)
    allmz = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    allin = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    prec = t.column('precursor_mz').to_numpy(zero_copy_only=False).astype(np.float64)
    add = np.asarray(t.column('adduct').cast(pa.string()).to_pylist(), dtype=object)
    ik  = np.asarray(t.column('inchikey14').cast(pa.string()).to_pylist(), dtype=object)
    smi = np.asarray(t.column('normalized_smiles').cast(pa.string()).to_pylist(), dtype=object)
    
    # Phase 2: Polarity extraction (True = positive, False = negative)
    imode = np.asarray(t.column('ionization_mode').cast(pa.string()).to_pylist(), dtype=object)
    is_pos = np.array([str(m).lower() == 'positive' for m in imode], dtype=bool)

    nm_meas = neutral_mass(prec, add)
    if CFG.STRUCT_MASS:
        # TIER-1: prefer the label's exact mass; fall back to the measured one where the
        # formula is unparseable (0.46% of rows) or contains an exotic element.
        formula   = np.asarray(t.column('molecular_formula').cast(pa.string()).to_pylist(), dtype=object)
        _fm       = {f: formula_mass(f) for f in set(formula.tolist())}
        nm_struct = np.array([_fm.get(f, np.nan) for f in formula], dtype=np.float64)
        nm        = np.where(np.isfinite(nm_struct), nm_struct, nm_meas)
        n_fixed   = int(np.isfinite(nm_struct).sum())
        print(f"[TIER-1] structure-derived neutral mass for {n_fixed:,}/{len(nm):,} spectra "
              f"({n_fixed/len(nm)*100:.1f}%); {len(nm)-n_fixed:,} fell back to precursor-derived.",
              flush=True)
    else:
        nm = nm_meas
    ok  = np.isfinite(nm)
    order = np.argsort(np.where(ok, nm, 1e18), kind='mergesort')
    best = {}
    for k, s in zip(ik, smi):
        if k and s and k not in best:
            best[k] = s
    print(f"[INFO] Reference Library loaded: {len(off)-1:,} spectra / {len(best):,} unique structures ({time.time()-t0:.1f}s)", flush=True)
    return dict(off=off, mz=allmz, it=allin, nm=nm, ik=ik, is_pos=is_pos, best=best,
                order=order, snm=nm[order], n_ok=int(ok.sum()))

def lib_window(L, target, tol):
    lo = np.searchsorted(L['snm'][:L['n_ok']], target - tol, 'left')
    hi = np.searchsorted(L['snm'][:L['n_ok']], target + tol, 'right')
    return L['order'][lo:hi]

def build_rep(L):
    """Phase 2: Extracts richest representative spectrum per unique (scaffold, polarity) pair."""
    npk = np.diff(L['off'])
    best = {}
    ik = L['ik']
    is_pos = L.get('is_pos', np.ones(len(ik), dtype=bool))
    if not getattr(CFG, 'POLARITY_ANALOG', True):
        is_pos = np.ones(len(ik), dtype=bool)   # one rep per scaffold (tier-1 behaviour)
    
    for i in range(len(ik)):
        k = ik[i]
        if k:
            rep_key_tuple = (k, bool(is_pos[i]))
            if rep_key_tuple not in best or npk[i] > npk[best[rep_key_tuple]]:
                best[rep_key_tuple] = i

    rep_idx = np.array(sorted(best.values()))
    nm = L['nm'][rep_idx]
    ok = np.isfinite(nm)
    rep_idx = rep_idx[ok]
    nm = nm[ok]
    rep_key = ik[rep_idx]
    rep_is_pos = is_pos[rep_idx]
    o = np.argsort(nm)
    
    print(f"[INFO] Built polarity-aware representative analog set: {len(rep_idx):,} spectra across {len(set(rep_key)):,} unique scaffolds.", flush=True)
    return rep_idx[o], rep_key[o], nm[o], rep_is_pos[o]


In [ ]:
# ===================================================================================
#  711,705-Structure Candidate Pool with Bitpacked Precomputed Fingerprints
# ===================================================================================
BITS = np.load(find_file('fp_bits.npy'))
_g = {}

def _fp_init():
    if not HAVE_RDKIT: return
    _g['m2'] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g['m3'] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g['rk'] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)

def fp_and_mass(smi):
    if not HAVE_RDKIT: return None
    if not _g: _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        fp = np.concatenate([
            _g['m2'].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g['m3'].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g['rk'].GetFingerprintAsNumPy(m).astype(np.uint8),
            np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)
        ])[BITS]
        return fp, float(ExactMolWt(m))
    except Exception:
        return None

class CandidatePool:
    """Unified candidate pool sorted by neutral exact mass."""
    def __init__(self, fp, mass, keys, smiles, nbits):
        o = np.argsort(mass)
        self._fp = fp[o]
        self.mass = mass[o]
        self.keys = np.asarray(keys, dtype=object)[o]
        self.smiles = np.asarray(smiles, dtype=object)[o]
        self.nbits = nbits
        self.k2i = {k: i for i, k in enumerate(self.keys)}

    def window(self, t, ppm):
        a = np.searchsorted(self.mass, t * (1 - ppm / 1e6), 'left')
        b = np.searchsorted(self.mass, t * (1 + ppm / 1e6), 'right')
        return np.arange(a, b)

    def fps(self, idx):
        return np.unpackbits(np.asarray(self._fp[idx]), axis=1)[:, :self.nbits]

def build_candidate_pool():
    t0 = time.time()
    d = os.path.dirname(find_file('coco_fp.npy'))
    cm = pickle.load(open(os.path.join(d, 'coco_meta.pkl'), 'rb'))
    co_fp = np.load(os.path.join(d, 'coco_fp.npy'))
    co_mass = np.load(os.path.join(d, 'coco_mass.npy'))
    co_keys = np.asarray(cm['keys'], dtype=object)
    co_smis = np.asarray(cm['smiles'], dtype=object)
    print(f"[INFO] COCONUT 2.0 component: {len(co_mass):,} structures loaded ({time.time()-t0:.1f}s)", flush=True)

    if CFG.USE_BIO_DB:
        try:
            bd = os.path.dirname(find_file('bio_fp.npy'))
            bm = pickle.load(open(os.path.join(bd, 'bio_meta.pkl'), 'rb'))
            bi_fp = np.load(os.path.join(bd, 'bio_fp.npy'))
            bi_mass = np.load(os.path.join(bd, 'bio_mass.npy'))
            co_fp = np.vstack([co_fp, bi_fp])
            co_mass = np.concatenate([co_mass, bi_mass])
            co_keys = np.concatenate([co_keys, np.asarray(bm['keys'], dtype=object)])
            co_smis = np.concatenate([co_smis, np.asarray(bm['smiles'], dtype=object)])
            print(f"[INFO] + ChEBI/LIPID MAPS: {len(bi_mass):,} structures integrated.", flush=True)
        except Exception:
            print("[INFO] Bio DB not found, continuing with COCONUT + Train.", flush=True)

    tr = pq.read_table(TRAIN, columns=['inchikey14', 'normalized_smiles']).to_pandas()
    tr = tr.dropna().drop_duplicates('inchikey14')
    coco_keys_set = set(co_keys)
    tr = tr[~tr.inchikey14.isin(coco_keys_set)]
    print(f"[INFO] Computing fingerprints for {len(tr):,} training structures (multiprocessing)...", flush=True)

    if HAVE_RDKIT:
        with MPool(4) as mp:
            res = mp.map(fp_and_mass, list(tr.normalized_smiles), chunksize=500)
        ok = [i for i, r in enumerate(res) if r is not None]
        tr_fp = np.packbits(np.stack([res[i][0] for i in ok]), axis=1)
        tr_mass = np.array([res[i][1] for i in ok])
        tr_keys = tr.inchikey14.values[ok]
        tr_smi = tr.normalized_smiles.values[ok]
    else:
        tr_fp = np.empty((0, co_fp.shape[1]), dtype=np.uint8)
        tr_mass = np.empty(0, dtype=np.float64)
        tr_keys = np.empty(0, dtype=object)
        tr_smi = np.empty(0, dtype=object)

    fp = np.vstack([co_fp, tr_fp])
    mass = np.concatenate([co_mass, tr_mass])
    keys = np.concatenate([co_keys, tr_keys])
    smis = np.concatenate([co_smis, tr_smi])
    good = np.isfinite(mass)

    pool = CandidatePool(fp[good], mass[good], keys[good], smis[good], cm['nbits'])
    print(f"[SUCCESS] Unified candidate pool ready: {len(pool.mass):,} structures ({time.time()-t0:.1f}s)", flush=True)
    return pool


In [ ]:
# ===================================================================================
#  Evidence Channels: Direct Library Match, Analog Match & MetFrag-lite
# ===================================================================================
AMU = {'C':12.0, 'H':1.00782503207, 'N':14.0030740048, 'O':15.9949146196, 'P':30.97376163,
       'S':31.97207100, 'F':18.99840322, 'Cl':34.96885268, 'Br':78.9183371, 'I':126.904473,
       'Na':22.9897692809, 'K':38.96370668, 'Si':27.9769265325, 'B':11.0093054, 'Se':79.9165213}
H_ATOM = AMU['H']
PROTON_MASS = H_ATOM - 0.00054857990

def clean_spectrum(mz, it):
    return _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                  CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)

def lib_sim(L, specs, target):
    """CLASS 1: Direct match against library spectra within tight neutral mass window."""
    cand = lib_window(L, target, target * CFG.PPM_WIN / 1e6)
    if len(cand) == 0: return {}
    agg = {}
    for mz, it in specs:
        qm, qp = clean_spectrum(mz, it)
        if len(qm) == 0: continue
        sc = search(qm, qp, cand, L['off'], L['mz'], L['it'],
                    CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)
        for c, s in zip(cand, sc):
            k = L['ik'][c]
            if s > agg.get(k, -1.0):
                agg[k] = float(s)
    return agg

def analog_sim(L, specs, target, rep, rep_key, rep_nm, rep_pos=None):
    """CLASS 2: Mass-shifted spectral entropy match across a wide +-200 Da window.

    Polarity is handled upstream: build_rep keeps the richest representative per
    (scaffold, polarity) pair. Candidates are NEVER masked out by query polarity,
    because agg[k] already takes the max over that scaffold's representatives - the
    better-matching polarity wins on its own. Coverage is therefore always >= the
    one-representative-per-scaffold behaviour it replaces, never less.
    """
    lo = np.searchsorted(rep_nm, target - CFG.ANALOG_WIN, 'left')
    hi = np.searchsorted(rep_nm, target + CFG.ANALOG_WIN, 'right')
    if hi <= lo: return []
    cand = rep[lo:hi]
    if len(cand) == 0: return []
    shift = (target - rep_nm[lo:hi]).astype(np.float32)
    ckey = rep_key[lo:hi]

    agg = {}
    for mz, it in specs:
        qm, qp = clean_spectrum(mz, it)
        if len(qm) == 0: continue
        sc = search_shift(qm, qp, cand, L['off'], L['mz'], L['it'],
                          CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                          CFG.INT_POWER, CFG.ENT_WEIGHT, shift)
        for c, k, s in zip(cand, ckey, sc):
            if s > agg.get(k, -1.0):
                agg[k] = float(s)
    return sorted(agg.items(), key=lambda x: -x[1])[:CFG.N_ANALOG]

# MetFrag-lite: In-silico 1-2 bond fragmentation
def mol_graph(smi):
    if not HAVE_RDKIT: return None
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    n = m.GetNumAtoms()
    w = np.zeros(n)
    for a in m.GetAtoms():
        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H_ATOM
    if (w == 0).any(): return None
    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]
    return w, bonds, n

def _components(n, bonds, drop):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(bonds):
        if i in drop: continue
        adj[a].append(b); adj[b].append(a)
    seen = np.zeros(n, bool); comps = []
    for s in range(n):
        if seen[s]: continue
        stack = [s]; seen[s] = True; cur = [s]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)
        comps.append(cur)
    return comps

def fragment_masses(smi, max_breaks=2, max_bonds=None):
    if max_bonds is None: max_bonds = CFG.MAX_BONDS
    g = mol_graph(smi)
    if g is None: return np.zeros(0)
    w, bonds, n = g
    nb = len(bonds)
    if nb == 0 or nb > max_bonds: return np.array([w.sum()])
    out = {w.sum()}
    for i in range(nb):
        for c in _components(n, bonds, {i}):
            out.add(float(w[c].sum()))
    if max_breaks >= 2:
        for i in range(nb):
            for j in range(i + 1, nb):
                for c in _components(n, bonds, {i, j}):
                    out.add(float(w[c].sum()))
    return np.array(sorted(out))

def explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):
    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0
    ion = []
    for dh in h_shifts:
        ion.append(frag_mass + dh * H_ATOM + (PROTON_MASS if mode > 0 else -PROTON_MASS))
    ion = np.sort(np.concatenate(ion))
    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()
    if tot <= 0: return 0.0
    idx = np.searchsorted(ion, peak_mz)
    ok = np.zeros(len(peak_mz), bool)
    for off in (-1, 0):
        k = np.clip(idx + off, 0, len(ion) - 1)
        ok |= np.abs(ion[k] - peak_mz) <= tol
    return float(w[ok].sum() / tot)

def _frag_masses_wrapper(smi):
    try: return fragment_masses(smi)
    except Exception: return np.zeros(0)

def frag_scores(cand_smiles, specs, mode, workers=4):
    if not HAVE_RDKIT:
        return np.zeros(len(cand_smiles), np.float32)
    with MPool(workers) as mp:
        frags = mp.map(_frag_masses_wrapper, cand_smiles, chunksize=8)
    peaks = []
    for mz, it in specs:
        m2, i2 = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                        CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)
        peaks.append((np.asarray(m2, float), np.asarray(i2, float)))
    out = np.zeros(len(cand_smiles), np.float32)
    for j, f in enumerate(frags):
        out[j] = max((explain_score(f, a, b, mode=mode, tol=CFG.MZ_TOL) for a, b in peaks), default=0.0)
    return out


In [ ]:
# ===================================================================================
#  Channel 4: Neural Spectrum -> Molecular Fingerprint Model (FPNet)
#  Ranking is f . z (Exact Bayes log-likelihood dot product)
# ===================================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

MAX_PEAKS_NN = 128
ADDUCT_LIST = ["[M+H]+","[M+NH4]+","[M+Na]+","[M+K]+","[M-H2O+H]+","[M-2H2O+H]+","[M]+",
               "[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+C2H4O2-H]-","[M+Cl]-","[M]-",
               "[M+2H]2+","[M-2H]-","[2M+H]+","[2M+Na]+","[2M+NH4]+","[2M-H]-","[2M+K]+",
               "[2M+CH2O2-H]-","[2M+C2H4O2-H]-","[2M+Na-2H]-","[M+Na-2H]-","[M-H2O]+","<unk>"]
ADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF", "Orbitrap", "QTOF", "IT", "other"]
INSTR_IX = {a: i for i, a in enumerate(INSTR_LIST)}

def instr_family(s):
    if s is None: return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4

def prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS_NN, floor=1e-3, win=50.0, per_win=8):
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = it >= floor * mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        order = np.argsort(-it)
        bucket = (mz // win).astype(np.int64)
        cnt = {}; sel = []
        for i in order:
            b = bucket[i]; c = cnt.get(b, 0)
            if c < per_win: cnt[b] = c + 1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            rest = np.array([i for i in order if i not in set(sel.tolist())])
            need = max_peaks - len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it / it.max())
    return mz.astype(np.float32), v.astype(np.float32)

class SinEmb(nn.Module):
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim // 2
        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)
        self.register_buffer('inv', (2 * math.pi) / wav)
    def forward(self, x):
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)

class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h = h
        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(drop)
    def forward(self, x, pad):
        B, N, D = x.shape; y = self.n1(x)
        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        m = (~pad)[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))
        return x + self.drop(self.ff(self.n2(x)))

class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d = d
        self.mz_emb = SinEmb(d)
        self.nl_emb = SinEmb(d)
        self.pk = nn.Linear(2 * d + 1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d + 3, d)
        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))

    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B, N = mz.shape
        nl = (prec[:, None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec),
                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:, 0]
        msk = (~pad[:, 1:]).float().unsqueeze(-1)
        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))

_MODEL = None

def load_neural_models():
    global _MODEL
    paths = sorted(glob.glob('/kaggle/input/**/fp_*.pt', recursive=True))
    if not paths:
        print("[INFO] No pretrained FPNet checkpoints found — proceeding with 3 channels.")
        return None
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    single, merged = [], []
    for pth in paths:
        ck = torch.load(pth, map_location='cpu', weights_only=False)
        net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).to(dev).eval()
        net.load_state_dict(ck['model'])
        if 'merged' in pth.split('/')[-1]:
            merged.append(net)
        else:
            single.append(net)
        print(f"  Loaded {os.path.basename(pth)}: d={ck['d']}, layers={ck['layers']}, step={ck.get('step')}")
    print(f"[SUCCESS] Neural FPNet Ensemble ready: {len(single)} single + {len(merged)} merged on {dev}")
    _MODEL = (single, merged, dev, ck['nbits'])
    return _MODEL

def _merge_peaks(sub):
    mz = np.concatenate([np.asarray(r.ms2_mzs, float) for r in sub.itertuples()])
    it = np.concatenate([np.asarray(r.ms2_normalized_intensities, float) /
                         max(float(np.asarray(r.ms2_normalized_intensities, float).max()), 1e-9)
                         for r in sub.itertuples()])
    o = np.argsort(mz); mz, it = mz[o], it[o]
    keep = np.ones(len(mz), bool)
    for j in range(1, len(mz)):
        if mz[j] - mz[j - 1] < 0.005:
            if it[j] >= it[j - 1]: keep[j - 1] = False
            else: keep[j] = False
    return mz[keep], it[keep]

@torch.no_grad()
def model_logits(sub):
    if _MODEL is None: return None
    single, merged, dev, nbits = _MODEL
    out = []
    if single:
        za = _logits_from(sub, single)
        if za is not None: out.append(za)
    if merged:
        mz, it = _merge_peaks(sub)
        r0 = next(sub.itertuples())
        zb = _logits_raw([(mz, it)], merged, float(np.median(sub.precursor_mz)), r0.adduct,
                         r0.instrument_type, 25.0,
                         float(np.mean([1.0 if m == 'positive' else -1.0 for m in sub.ionization_mode])))
        if zb is not None: out.append(zb)
    return np.mean(out, axis=0) if out else None

@torch.no_grad()
def _logits_from(sub, nets):
    if _MODEL is None: return None
    dev = _MODEL[2]
    rows = list(sub.itertuples())
    P = [prep_peaks(r.ms2_mzs, r.ms2_normalized_intensities, float(r.precursor_mz)) for r in rows]
    P = [(a, b) for a, b in P if len(a)]
    if not P: return None
    B = len(P); N = max(len(a) for a, _ in P)
    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)
    for i, (a, b) in enumerate(P):
        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False
    def ce_of(r):
        v = r.collision_energy_ev
        try: return float(np.mean(np.atleast_1d(v))) if v is not None and len(np.atleast_1d(v)) else 25.0
        except Exception: return 25.0
    T = lambda x: torch.as_tensor(x, device=dev)
    args = (T(mz), T(it), T(pad),
            T(np.array([float(r.precursor_mz) for r in rows[:B]], np.float32)),
            T(np.array([ADDUCT_IX.get(r.adduct, ADDUCT_IX['<unk>']) for r in rows[:B]])),
            T(np.array([instr_family(r.instrument_type) for r in rows[:B]])),
            T(np.array([ce_of(r) for r in rows[:B]], np.float32)),
            T(np.array([1.0 if r.ionization_mode == 'positive' else -1.0 for r in rows[:B]], np.float32)))
    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)

@torch.no_grad()
def _logits_raw(pairs, nets, prec, adduct, instrument, ce, mode):
    if _MODEL is None: return None
    dev = _MODEL[2]
    P = [prep_peaks(mz, it, prec) for mz, it in pairs]
    P = [(a, b) for a, b in P if len(a)]
    if not P: return None
    B = len(P); N = max(len(a) for a, _ in P)
    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)
    for i, (a, b) in enumerate(P):
        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False
    T = lambda x: torch.as_tensor(x, device=dev)
    args = (T(mz), T(it), T(pad), T(np.full(B, prec, np.float32)),
            T(np.full(B, ADDUCT_IX.get(adduct, ADDUCT_IX['<unk>']))),
            T(np.full(B, instr_family(instrument))),
            T(np.full(B, ce, np.float32)), T(np.full(B, mode, np.float32)))
    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)


In [ ]:
# ===================================================================================
#  31-Feature Extraction & Calibrated Dual-Prior Seed-Ensemble Ranker
# ===================================================================================
from sklearn.ensemble import HistGradientBoostingClassifier

N_ANALOG = CFG.N_ANALOG
P_SIM    = CFG.SIM_POWER
N_FEAT   = 31

def _rank_norm(x):
    o = np.argsort(-x)
    r = np.empty(len(x))
    r[o] = np.arange(len(x))
    return r / max(1, len(x) - 1)

def _z(x):
    s = x.std()
    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)

def rank_features(cand_fp, cand_lib, analog_fp, analog_sim_vals, model_logits=None, frag=None):
    nc = cand_fp.shape[0]
    cf = cand_fp.astype(np.float32)
    cs = cf.sum(1)
    lv = np.asarray(cand_lib, np.float32)
    lvmax = float(lv.max()) if nc else 0.0

    if analog_fp is not None and len(analog_sim_vals):
        af = analog_fp.astype(np.float32)
        asum = af.sum(1)
        inter = cf @ af.T
        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)
        w = np.clip(np.asarray(analog_sim_vals, np.float32), 0, None)
        ap = (tan * (w ** P_SIM)[None, :]).max(1)
        a1 = (tan * w[None, :]).max(1)
        best_tan = tan.max(1)
        top_tan = tan[:, 0]
        top_sim = float(w[0])
        mean_tan = (tan * (w ** P_SIM)[None, :]).sum(1) / ((w ** P_SIM).sum() + 1e-9)
    else:
        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32)
        top_sim = 0.0

    apmax = float(ap.max()) if nc else 0.0

    if model_logits is not None:
        raw = cf @ np.asarray(model_logits, np.float32)
        nrm = raw / np.sqrt(np.maximum(cs, 1.0))
        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),
                 (raw == raw.max()).astype(np.float32)]
    else:
        mfeat = [np.zeros(nc, np.float32)] * 6

    if model_logits is not None and nc:
        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))
        lbest = int(np.argmax(lv)) if lvmax > 0 else -1
        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0
        abest = int(np.argmax(ap)) if apmax > 0 else -1
        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0
        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),
                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]
    else:
        xfeat = [np.zeros(nc, np.float32)] * 6

    if frag is not None:
        fr = np.asarray(frag, np.float32)
        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]
    else:
        ffeat = [np.zeros(nc, np.float32)] * 4

    return np.column_stack([
        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),
        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,
        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),
        np.full(nc, np.log(max(nc, 1))),
        *mfeat, *ffeat, *xfeat,
    ]).astype(np.float32)

In [ ]:
# ===================================================================================
#  Stage-D config + explicit artifact selection
# ===================================================================================
class SD:
    VAL_PARQUET = None      # auto-discovered
    TOPN = 25
    OUT = '/kaggle/working'

NFEAT = N_FEAT
SD.VAL_PARQUET = find_file('val.parquet')
VAL_KEY_PATH    = find_file('val_key.pkl')

all_pt  = sorted(glob.glob('/kaggle/input/**/fp_*.pt', recursive=True))
all_npz = sorted(glob.glob('/kaggle/input/**/rank_train.npz', recursive=True))

def _is_new(p):
    try:
        return torch.load(p, map_location='cpu', weights_only=False).get('fp_bits_sha1') is not None
    except Exception:
        return False

NEW_PT = [p for p in all_pt if _is_new(p)]
OLD_PT = [p for p in all_pt if p not in NEW_PT]

# The new matrix is the one that SHIPS WITH the new checkpoints, so identify it by
# directory co-location rather than by a dataset name the user might have typed differently.
_new_dirs = {os.path.dirname(p) for p in NEW_PT}
NEW_NPZ = [p for p in all_npz if os.path.dirname(p) in _new_dirs]
if not NEW_NPZ:   # fall back to the name, then tell the user what went wrong
    NEW_NPZ = [p for p in all_npz if 'fpnet' in p.lower() and 'v3' in p.lower()]
OLD_NPZ = [p for p in all_npz if p not in NEW_NPZ]

print("OLD checkpoints:", [os.path.basename(p) for p in OLD_PT])
print("NEW checkpoints:", [os.path.basename(p) for p in NEW_PT])
print("OLD matrix     :", OLD_NPZ)
print("NEW matrix     :", NEW_NPZ)
assert OLD_PT, "no OLD checkpoints found - attach casmi26-fp-models-v2"
assert NEW_PT, "no NEW checkpoints found - attach casmi26-fpnet-v3 (they carry fp_bits_sha1)"
assert OLD_NPZ, "no OLD rank_train.npz - attach casmi26-ranker-features"
assert NEW_NPZ, (
    "no NEW rank_train.npz found next to the new checkpoints. Put the Stage-C output in the "
    "SAME Kaggle dataset as the checkpoints it was built from - they are a matched pair.")
assert len(NEW_NPZ) == 1 and len(OLD_NPZ) == 1, f"ambiguous matrices: {OLD_NPZ} / {NEW_NPZ}"

In [ ]:
# ===================================================================================
#  Explicit model / ranker loaders (select by path, never by glob)
# ===================================================================================
def load_models(paths):
    """load_neural_models() with an explicit file list instead of a glob over all inputs."""
    global _MODEL
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    single, merged, ck = [], [], None
    for pth in sorted(paths):
        ck = torch.load(pth, map_location='cpu', weights_only=False)
        net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).to(dev).eval()
        net.load_state_dict(ck['model'])
        (merged if 'merged' in os.path.basename(pth) else single).append(net)
    assert single and merged, f"need >=1 single and >=1 merged in {paths}"
    _MODEL = (single, merged, dev, ck['nbits'])
    print(f"  loaded {len(single)} single + {len(merged)} merged on {dev}")
    return _MODEL

def fit_rankers(npz_path):
    """The RANKERS fit from phase1, parameterised by which matrix it reads."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    z = np.load(npz_path)
    out = []
    for w1 in CFG.W1_PRIORS:
        W = np.where(z['M'] == 0, w1, 1.0 - w1)
        for sd in CFG.SEEDS:
            m = HistGradientBoostingClassifier(random_state=sd, **CFG.GBM)
            m.fit(z['X'], z['Y'], sample_weight=W)
            out.append(m)
    print(f"  {os.path.basename(npz_path)}: {len(out)} GBMs on {z['X'].shape[0]:,} x {z['X'].shape[1]}")
    assert z['X'].shape[1] == NFEAT, f"matrix has {z['X'].shape[1]} features, rank_features makes {NFEAT}"
    return out

def proba(rankers, X):
    return np.mean([m.predict_proba(X)[:, 1] for m in rankers], axis=0)

In [ ]:
# ===================================================================================
#  Ban-aware channels (identical to Stage C)
# ===================================================================================
def _drop(cand, ban_rows, ban_keys, ik_arr):
    if cand.size == 0 or (not len(ban_rows) and not len(ban_keys)): return cand
    m = np.ones(len(cand), bool)
    if len(ban_rows): m &= ~np.isin(cand, ban_rows)
    if len(ban_keys): m &= ~np.isin(ik_arr[cand], ban_keys)
    return cand[m]

def lib_sim_ban(L, specs, target, ban_rows=(), ban_keys=()):
    cand = lib_window(L, target, target * CFG.PPM_WIN / 1e6)
    cand = _drop(cand, np.asarray(ban_rows, np.int64), np.asarray(ban_keys, dtype=object), L['ik'])
    if len(cand) == 0: return {}
    agg = {}
    for mz, it in specs:
        qm, qp = clean_spectrum(mz, it)
        if len(qm) == 0: continue
        sc = search(qm, qp, cand, L['off'], L['mz'], L['it'],
                    CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)
        for c, s in zip(cand, sc):
            k = L['ik'][c]
            if s > agg.get(k, -1.0): agg[k] = float(s)
    return agg

def analog_sim_ban(L, specs, target, rep, rep_key, rep_nm, ban_rows=(), ban_keys=()):
    lo = np.searchsorted(rep_nm, target - CFG.ANALOG_WIN, 'left')
    hi = np.searchsorted(rep_nm, target + CFG.ANALOG_WIN, 'right')
    if hi <= lo: return []
    cand  = rep[lo:hi]
    shift = (target - rep_nm[lo:hi]).astype(np.float32)
    ckey  = rep_key[lo:hi]
    if len(ban_rows) or len(ban_keys):
        m = np.ones(len(cand), bool)
        if len(ban_rows): m &= ~np.isin(cand, np.asarray(ban_rows, np.int64))
        if len(ban_keys): m &= ~np.isin(ckey, np.asarray(ban_keys, dtype=object))
        cand, shift, ckey = cand[m], shift[m], ckey[m]
    if len(cand) == 0: return []
    agg = {}
    for mz, it in specs:
        qm, qp = clean_spectrum(mz, it)
        if len(qm) == 0: continue
        sc = search_shift(qm, qp, cand, L['off'], L['mz'], L['it'],
                          CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                          CFG.INT_POWER, CFG.ENT_WEIGHT, shift)
        for c, k, s in zip(cand, ckey, sc):
            if s > agg.get(k, -1.0): agg[k] = float(s)
    return sorted(agg.items(), key=lambda x: -x[1])[:CFG.N_ANALOG]

_MP = None
def frag_scores_fast(cand_smiles, specs, mode):
    global _MP
    if not HAVE_RDKIT: return np.zeros(len(cand_smiles), np.float32)
    if _MP is None: _MP = MPool(4)
    frags = _MP.map(_frag_masses_wrapper, cand_smiles, chunksize=8)
    peaks = []
    for mz, it in specs:
        m2, i2 = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                        CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)
        peaks.append((np.asarray(m2, float), np.asarray(i2, float)))
    out = np.zeros(len(cand_smiles), np.float32)
    for j, f in enumerate(frags):
        out[j] = max((explain_score(f, a, b, mode=mode, tol=CFG.MZ_TOL) for a, b in peaks), default=0.0)
    return out
print("ban-aware channels ready")

In [ ]:
# ===================================================================================
#  Load the engine and the validation set
# ===================================================================================
pool = build_candidate_pool()
L = load_library(TRAIN)
rep, rep_key, rep_nm, rep_pos = build_rep(L)

te = pq.read_table(SD.VAL_PARQUET).to_pandas()
te['nm'] = neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)
mols = list(te.groupby('molecule_id'))

key = pickle.load(open(VAL_KEY_PATH, 'rb'))
TRUTH  = {k['molecule_id']: k['inchikey14'] for k in key}
COHORT = {k['molecule_id']: k['cohort']     for k in key}
BAN    = {k['molecule_id']: (np.asarray(k['ban_rows'], np.int64),
                             np.asarray(k['ban_keys'], dtype=object)) for k in key}
print(f"[INFO] {len(mols):,} molecules | cohorts {pd.Series(COHORT).value_counts().to_dict()}")

# This is the blind spot to keep in view: every val truth comes from train.parquet, which is
# inside the pool, so pool presence is ~1.0 by construction. Real Class-3 test molecules are
# not in the pool at all and score 0 regardless of model quality. Val MRR will always read
# above the leaderboard; the gap is the pool miss rate, and nothing here can measure it.
in_pool = pd.Series({m: (TRUTH[m] in pool.k2i) for m, _ in mols})
print("truth-in-pool rate by cohort:")
print(pd.DataFrame({'in_pool': in_pool, 'cohort': pd.Series(COHORT)})
      .groupby('cohort').in_pool.mean().round(4).to_string())

In [ ]:
# ===================================================================================
#  Leakage assertion - the same check the validation-set notebook makes
# ===================================================================================
mid1 = next(m for m in COHORT if COHORT[m] == 'C2')
sub1 = te[te.molecule_id == mid1]
specs1 = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub1.itertuples()]
tgt1 = float(np.median(sub1.nm.values[np.isfinite(sub1.nm.values)])) / (1.0 + CFG.PPM_CENTER / 1e6)
br1, bk1 = BAN[mid1]
t1 = TRUTH[mid1]
banned   = lib_sim_ban(L, specs1, tgt1, br1, bk1).get(t1, 0.0)
unbanned = lib_sim_ban(L, specs1, tgt1, [], []).get(t1, 0.0)
print(f"Class-2 self-match  with ban: {banned:.4f}   (must be ~0)")
print(f"Class-2 self-match  no  ban: {unbanned:.4f}   (leakage if this were used)")
assert banned < 0.2, "ban filter is not working - every number below would be meaningless"
print("ban filter verified")

In [ ]:
# ===================================================================================
#  PASS 1 - channels 1/2/3, computed once and cached (config-independent)
# ===================================================================================
t0 = time.time()
CACHE = {}
for gi, (mid, sub) in enumerate(mols):
    nms = sub.nm.values[np.isfinite(sub.nm.values)]
    if len(nms) == 0:
        CACHE[mid] = None; continue
    target = float(np.median(nms)) / (1.0 + CFG.PPM_CENTER / 1e6)
    specs  = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]
    br, bk = BAN[mid]

    lib_hits = lib_sim_ban(L, specs, target, br, bk)
    analogs  = analog_sim_ban(L, specs, target, rep, rep_key, rep_nm, br, bk)
    ion_mode = float(np.mean([1.0 if m == 'positive' else -1.0 for m in sub.ionization_mode]))

    cand = pool.window(target, CFG.PPM_WIN)
    if len(cand) == 0: cand = pool.window(target, CFG.PPM_FALLBACK)

    ids, sims = [], []
    for k, s in analogs:
        i = pool.k2i.get(k, -1)
        if i >= 0: ids.append(i); sims.append(s)

    ent = dict(target=target, cand=cand, ion=ion_mode, ids=np.array(ids, np.int64),
               sims=np.array(sims, np.float32),
               lv=np.array([lib_hits.get(pool.keys[c], 0.0) for c in cand], np.float32),
               fsc=frag_scores_fast([pool.smiles[c] for c in cand], specs, ion_mode) if len(cand) else np.zeros(0, np.float32),
               lib_hits=lib_hits, tiers={})

    # Whether stage B fires depends only on the number of DISTINCT keys in the tight window,
    # which is independent of the model and the ranker -- so its inputs can be cached too.
    if getattr(CFG, 'EXPAND_BACKFILL', True) and len(set(pool.keys[cand].tolist())) < SD.TOPN:
        for tier in CFG.PPM_EXPAND[1:]:
            wide = pool.window(target, tier)
            new_cand = wide[~np.isin(wide, cand)]
            if len(new_cand) == 0: continue
            ent['tiers'][tier] = dict(
                cand=new_cand,
                lv=np.array([lib_hits.get(pool.keys[c], 0.0) for c in new_cand], np.float32),
                fsc=frag_scores_fast([pool.smiles[c] for c in new_cand], specs, ion_mode))
            if len(set(pool.keys[np.concatenate([cand, new_cand])].tolist())) >= SD.TOPN:
                break
    CACHE[mid] = ent
    if (gi + 1) % 50 == 0 or gi == len(mols) - 1:
        el = time.time() - t0
        print(f"  pass1 {gi+1}/{len(mols)} | {el:.0f}s | eta {(len(mols)-gi-1)*el/(gi+1)/60:.0f} min", flush=True)
print(f"[INFO] pass 1 done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ===================================================================================
#  PASS 2 - Channel-4 logits, once per model set
# ===================================================================================
ZLOG = {}
for tag, paths in (('old', OLD_PT), ('new', NEW_PT)):
    print(f"[{tag}]"); load_models(paths)
    t0 = time.time(); z = {}
    for gi, (mid, sub) in enumerate(mols):
        z[mid] = model_logits(sub) if CACHE[mid] is not None else None
        if (gi + 1) % 200 == 0:
            print(f"  logits {gi+1}/{len(mols)} | {time.time()-t0:.0f}s", flush=True)
    ZLOG[tag] = z
    print(f"  {tag} logits done in {time.time()-t0:.0f}s")

In [ ]:
# ===================================================================================
#  PASS 3 - assemble features and score, four ways
# ===================================================================================
RANKERS = {'old': fit_rankers(OLD_NPZ[0]), 'new': fit_rankers(NEW_NPZ[0])}

def score_set(cand, lv, fsc, ent, zlog, rankers):
    cfp = pool.fps(cand)
    afp = pool.fps(ent['ids']) if len(ent['ids']) else None
    X = rank_features(cfp, lv, afp, ent['sims'], zlog, fsc)[:, :NFEAT]
    return np.argsort(-proba(rankers, X))

def rank_of_truth(mid, mtag, rtag):
    ent = CACHE[mid]
    if ent is None: return 0
    truth = TRUTH[mid]
    zlog = ZLOG[mtag][mid]
    rk = RANKERS[rtag]
    keys_out, seen = [], set()

    def take(order, cand):
        for i in order:
            k = pool.keys[cand[i]]
            if k is not None and k not in seen and len(str(k)) >= 14:
                seen.add(k); keys_out.append(k)
                if len(keys_out) == SD.TOPN: return True
        return False

    if len(ent['cand']):
        if take(score_set(ent['cand'], ent['lv'], ent['fsc'], ent, zlog, rk), ent['cand']):
            return (keys_out.index(truth) + 1) if truth in keys_out else 0
    for tier, t in ent['tiers'].items():                       # stage B: append-only
        if len(keys_out) >= SD.TOPN: break
        if take(score_set(t['cand'], t['lv'], t['fsc'], ent, zlog, rk), t['cand']):
            break
    if len(keys_out) < SD.TOPN:                                # stage C: mass-nearest top-up
        wide = pool.window(ent['target'], getattr(CFG, 'BACKFILL_PPM', 120.0))
        if len(wide):
            for c in wide[np.argsort(np.abs(pool.mass[wide] - ent['target']))]:
                k = pool.keys[c]
                if k is not None and k not in seen and len(str(k)) >= 14:
                    seen.add(k); keys_out.append(k)
                    if len(keys_out) == SD.TOPN: break
    return (keys_out.index(truth) + 1) if truth in keys_out else 0

CONFIGS = {'base': ('old','old'), 'model_only': ('new','old'),
           'matrix_only': ('old','new'), 'both': ('new','new')}
RES = {}
for name, (mtag, rtag) in CONFIGS.items():
    t0 = time.time()
    rows = [dict(mid=mid, cohort=COHORT[mid], rank=rank_of_truth(mid, mtag, rtag))
            for mid, _ in mols]
    d = pd.DataFrame(rows)
    d['rr'] = np.where(d['rank'] > 0, 1.0 / d['rank'], 0.0)
    RES[name] = d
    print(f"  {name:12s} MRR {d.rr.mean():.4f}  ({time.time()-t0:.0f}s)")

In [ ]:
# ===================================================================================
#  Report + the decision rule
# ===================================================================================
def summarise(d):
    g = d.groupby('cohort').rr
    t = pd.DataFrame({'n': g.size(), 'MRR': g.mean(),
                      'top1': d.groupby('cohort')['rank'].apply(lambda r: (r == 1).mean()),
                      'hit25': d.groupby('cohort')['rank'].apply(lambda r: (r > 0).mean()),
                      'se': g.std() / np.sqrt(g.size())})
    t.loc['ALL'] = [len(d), d.rr.mean(), (d['rank'] == 1).mean(), (d['rank'] > 0).mean(),
                    d.rr.std() / np.sqrt(len(d))]
    return t.round(4)

for name, d in RES.items():
    print(f"\n=== {name} ===")
    print(summarise(d).to_string())

base = RES['base'].set_index('mid')
print("\n\n=== deltas vs base (MRR, by cohort) ===")
rowsd = []
for name in ['model_only', 'matrix_only', 'both']:
    cur = RES[name].set_index('mid')
    dd = (cur.rr - base.rr)
    r = {'config': name}
    for c in sorted(base.cohort.unique()):
        m = base.cohort == c
        r[c] = round(dd[m].mean(), 4)
    r['ALL'] = round(dd.mean(), 4)
    # paired SE: the same molecules under both configs, so difference-of-paired is the
    # right error bar, and it is much tighter than two independent SEs
    r['se_paired'] = round(dd.std() / np.sqrt(len(dd)), 4)
    rowsd.append(r)
print(pd.DataFrame(rowsd).to_string(index=False))

# ---- decision rule ----------------------------------------------------------------
b = RES['both'].set_index('mid'); dd = b.rr - base.rr
def cohort_delta(c):
    m = base.cohort == c
    v = dd[m]
    return v.mean(), v.std() / np.sqrt(max(len(v), 1))

print("\n=== SHIP / HOLD ===")
ok = True
for c in ['C2', 'C2_hard']:
    if c not in set(base.cohort): continue
    mu, se = cohort_delta(c)
    good = mu > se
    ok &= good
    print(f"  {c:9s} delta {mu:+.4f} +- {se:.4f}  -> {'gain' if good else 'NOT above 1 SE'}")
if 'C1_npx' in set(base.cohort):
    mu, se = cohort_delta('C1_npx')
    good = mu > -se
    ok &= good
    print(f"  {'C1_npx':9s} delta {mu:+.4f} +- {se:.4f}  -> {'no regression' if good else 'REGRESSED past 1 SE'}")
print(f"\n  {'SHIP `both`' if ok else 'HOLD - the gain is not separable from noise'}")

pd.concat([d.assign(config=n) for n, d in RES.items()]).to_csv(
    os.path.join(SD.OUT, 'stage_d_results.csv'), index=False)
print(f"\nper-molecule results -> {SD.OUT}/stage_d_results.csv")